In [1]:
import json
import os
import urllib
import ssl

def download_and_load_file(file_path, url):
    ssl_context = ssl.create_default_context()
    ssl_context.check_hostname = False
    ssl_context.verify_mode = ssl.CERT_NONE

    if not os.path.exists(file_path):
        with urllib.request.urlopen(url, context=ssl_context) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data


file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))


Number of entries: 1100


In [2]:
print("Example_entry:\n",data[50])

Example_entry:
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


In [3]:
def format_input(entry):
    instruction_test=(
        f"Below is an instruction that describes a task."
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction :\n{entry['instruction']}" 
    )
    input_text=f"\n\n### Input:\n {entry['input']}" if entry['input'] else ""
    return instruction_test+input_text


In [4]:
model_input=format_input(data[50])
desired_response=f"\n\n### Response:\n {data[50]['output']}"
print(model_input+desired_response)

Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction :
Identify the correct spelling of the following word.

### Input:
 Ocassion

### Response:
 The correct spelling is 'Occasion.'


In [5]:
train_portion=int(len(data)*0.85)
test_portion=int(len(data)*0.1)
val_portion=len(data)-train_portion-test_portion
train_data=data[:train_portion]
test_data=data[train_portion:train_portion+test_portion]
val_data=data[train_portion+test_portion:]
print(f"{len(train_data)}\n{len(val_data)}\n{len(test_data)}")

935
55
110


In [6]:
import tiktoken


In [7]:
import torch
from torch.utils.data import Dataset,DataLoader

class InstructionDataset(Dataset):
    def __init__(self,data,tokenizer):
        self.data=data
        self.encoded_texts=[]

        for entry in data:
            instruction_plus_input=format_input(entry)
            response_text=f"\n\n### Response:\n{entry['output']}"
            full_text=instruction_plus_input+response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )
    def __getitem__(self, index):
        return self.encoded_texts[index]
    def __len__(self):
        return len(self.data)
        


In [8]:
tokenizer=tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>",allowed_special={"<|endoftext|>"}))

[50256]


In [9]:
def custom_collate_draft_1(
        batch,
        pad_token_id=50256,
        device="cpu"
):
    batch_max_length=max(len(item)+1 for item in batch)
    inputs_list=[]
    for item in batch:
        new_item=item.copy()
        new_item+=[pad_token_id]
        padded=(
            new_item+[pad_token_id]*(batch_max_length-len(new_item))
        )
        inputs=torch.tensor(padded[:-1])
        inputs_list.append(inputs)
    inputs_tensor=torch.stack(inputs_list).to(device)
    return inputs_tensor

In [10]:
inputs_1=[0,1,2,3,4]
inputs_2=[5,6]
inputs_3=[7,8,9]
batch=(
    inputs_1,
    inputs_2,
    inputs_3
)
print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


In [11]:
def custom_collate_draft_2(   batch,
        pad_token_id=50256,
        device="cpu"
):
    batch_max_length=max(len(item)+1 for item in batch)
    inputs_list,targets_list=[],[]
    for item in batch:
        new_item=item.copy()
        new_item+=[pad_token_id]
        padded=(
            new_item+[pad_token_id]*(batch_max_length-len(new_item))
        )
        inputs=torch.tensor(padded[:-1])
        inputs_list.append(inputs)
        targets=torch.tensor(padded[1:])
        targets_list.append(targets)
    inputs_tensor=torch.stack(inputs_list).to(device)
    targets_tensor=torch.stack(targets_list).to(device)
    return inputs_tensor,targets_tensor

In [12]:
def custom_collate_fn(   batch,
        pad_token_id=50256,
        device="cpu",
        ignore_index=-100,
        allowed_max_length=None

):
    batch_max_length=max(len(item)+1 for item in batch)
    inputs_list,targets_list=[],[]
    for item in batch:
        new_item=item.copy()
        new_item+=[pad_token_id]
        padded=(
            new_item+[pad_token_id]*(batch_max_length-len(new_item))
        )
        inputs=torch.tensor(padded[:-1])
        
        targets=torch.tensor(padded[1:])
        mask=targets==pad_token_id
        indices=torch.nonzero(mask).squeeze()
        if indices.numel()>1:
            targets[indices[1:]]=ignore_index
        if allowed_max_length is not None:
            inputs=inputs[:allowed_max_length]
            targets=targets[:allowed_max_length]
        inputs_list.append(inputs)
        targets_list.append(targets)
    inputs_tensor=torch.stack(inputs_list).to(device)
    targets_tensor=torch.stack(targets_list).to(device)
    return inputs_tensor,targets_tensor

In [13]:
inputs_1=[0,1,2,3,4]
inputs_2=[5,6]
inputs_3=[7,8,9]
batch=(
    inputs_1,
    inputs_2,
    inputs_3
)
ipt,trt=custom_collate_fn(batch=batch)
print(ipt)
print(trt)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


In [14]:
logits=torch.tensor(
    [[-1.0,1.0],
      [-0.5,1.5]]
)
targets_1=torch.tensor([0,1])
loss_1=torch.nn.functional.cross_entropy(logits,targets_1)
print(loss_1)

tensor(1.1269)


In [15]:
logits_2=torch.tensor(
    [[-1.0,1.0],
     [-0.5,1.5],
     [-0.5,1.5]]
)
targets_2=torch.tensor([0,1,1])
loss_2=torch.nn.functional.cross_entropy(logits_2,targets_2)
print(loss_2)

tensor(0.7936)


In [16]:
from functools import partial
customized_collate=partial(custom_collate_fn,allowed_max_length=1024)

In [17]:
torch.manual_seed(123)
train_dataset=InstructionDataset(train_data,tokenizer)
train_loader=DataLoader(
    train_dataset,
    batch_size=8,
    collate_fn=customized_collate,
    shuffle=True,
    drop_last=True
)

In [18]:
val_dataset=InstructionDataset(val_data,tokenizer)
val_loader=DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=custom_collate_fn,
)
test_dataset=InstructionDataset(test_data,tokenizer)
test_loader=DataLoader(
    test_dataset,
    collate_fn=custom_collate_fn,
    shuffle=False,
    batch_size=8
)

In [19]:
from gpt_download3 import download_and_load_gpt2



Instructions for updating:
non-resource variables are not supported in the long term


In [20]:

settings,params=download_and_load_gpt2(model_size="124M",models_dir="gpt2")

c:\Users\kanha\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2\124M\checkpoint


c:\Users\kanha\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2\124M\encoder.json


c:\Users\kanha\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

File already exists and is up-to-date: gpt2\124M\model.ckpt.index


c:\Users\kanha\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2\124M\model.ckpt.meta


c:\Users\kanha\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2\124M\vocab.bpe


In [21]:
params.keys()

dict_keys(['blocks', 'ln_f', 'wpe', 'wte'])

In [22]:
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)


In [23]:
import torch.nn as nn

In [24]:
class LayerNorm(nn.Module):
  def __init__(self,emb_dim):
    super().__init__()
    self.eps=1e-5
    self.scale=nn.Parameter(torch.ones(emb_dim))
    self.shift=nn.Parameter(torch.zeros(emb_dim))
  def forward(self,x):
    mean=x.mean(dim=-1,keepdim=True)
    var=x.var(dim=-1,keepdim=True,unbiased=True)
    norm_x=(x-mean)/torch.sqrt(var+self.eps)
    return self.scale*norm_x+self.shift

In [25]:
import math

In [26]:
class GELU(nn.Module):
  def __init__(self):
    super().__init__()
  def forward(self, x):
        return 0.5 * x * (
            1 + torch.tanh(
                math.sqrt(2.0 / math.pi) * (x + 0.044715 * x**3)
            )
        )

In [27]:
class FeedForward(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.layers=nn.Sequential(
        nn.Linear(cfg["emb_dim"],4*cfg["emb_dim"]),
        GELU(),
        nn.Linear(4*cfg["emb_dim"],cfg["emb_dim"])
    )
  def forward(self,x):
    return self.layers(x)

In [28]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )
    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec


In [29]:
class TransformerBlock(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.att=MultiHeadAttention(
      d_in=cfg["emb_dim"],
      d_out=cfg["emb_dim"],
      context_length=cfg["context_length"],
      num_heads=cfg["n_heads"],
      dropout=cfg["drop_rate"],
      qkv_bias=cfg["qkv_bias"])
    self.ff=FeedForward(cfg)
    self.norm1=LayerNorm(cfg["emb_dim"])
    self.norm2=LayerNorm(cfg["emb_dim"])
    self.drop_shortcut=nn.Dropout(cfg["drop_rate"])
  def forward(self,x):
    shortcut=x
    x=self.norm1(x)
    x=self.att(x)
    x=self.drop_shortcut(x)
    x+=shortcut
    shortcut=x
    x=self.norm2(x)
    x=self.ff(x)
    x=self.drop_shortcut(x)
    x+=shortcut
    return x

In [30]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


In [31]:
def text_token(text,tokenizer):
  encoded=tokenizer.encode(text,allowed_special={'<|endoftext|>'})
  encoded_tensor=torch.tensor(encoded).unsqueeze(0)
  return encoded_tensor
def token_text(token_ids,tokenizer):
  flat=token_ids.squeeze(0)
  return tokenizer.decode(flat.tolist())


In [32]:
Basic_config={
    "vocab_size":50257,
    "context_length":1024,
    "drop_rate":0.0,
    "qkv_bias":True
}


In [33]:
model_config={
    "gpt2-small {124M}":{"emb_dim":768,"n_layers":12,"n_heads":12},
    "gpt2-medium {355M}":{"emb_dim":1024,"n_layers":24,'n_heads':16},
    "gpt2-large {774M}":{"emb_dim":1280,"n_layers":36,"n_heads":20},
    "gpt2-xl {1558M}":{"emb_dim":1600,"n_layers":48,"n_heads":25}

}

In [34]:
model_config = {
    "gpt2-small {124M}": {"emb_dim":768,"n_layers":12,"n_heads":12},
    "gpt2-medium {355M}": {"emb_dim":1024,"n_layers":24,"n_heads":16},
    "gpt2-large {774M}": {"emb_dim":1280,"n_layers":36,"n_heads":20},
    "gpt2-xl {1558M}": {"emb_dim":1600,"n_layers":48,"n_heads":25}
}

Choose = "gpt2-medium {355M}"

Basic_config.update(model_config[Choose])

model_size = Choose.split(" ")[-1].lstrip("{").rstrip("}")

settings, params = download_and_load_gpt2(
    model_size=model_size,
    models_dir="gpt2",
)

model = GPTModel(Basic_config)

c:\Users\kanha\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2\355M\checkpoint


c:\Users\kanha\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2\355M\encoder.json


c:\Users\kanha\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

File already exists and is up-to-date: gpt2\355M\model.ckpt.index


c:\Users\kanha\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2\355M\model.ckpt.meta


c:\Users\kanha\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File already exists and is up-to-date: gpt2\355M\vocab.bpe


In [35]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

In [36]:
import numpy as np

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])
    
    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight, 
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias, 
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight, 
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias, 
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight, 
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias, 
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale, 
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift, 
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale, 
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift, 
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["ln_f"]["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["ln_f"]["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])



In [37]:
load_weights_into_gpt(model,params)

In [38]:
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(i

In [39]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):

    # For-loop is the same as before: Get logits, and only focus on last time step
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # New: Filter logits with top_k sampling
        if top_k is not None:
            # Keep only top_k values
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        # New: Apply temperature scaling
        if temperature > 0.0:
            logits = logits / temperature

            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        # Otherwise same as before: get idx of the vocab entry with the highest logits value
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if idx_next == eos_id:  # Stop generating early if end-of-sequence token is encountered and eos_id is specified
            break

        # Same as before: append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

In [40]:
torch.manual_seed(123)
input_text=format_input(val_data[0])
print(input_text)

Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction :
Convert the active sentence to passive: 'The chef cooks the meal every day.'


In [40]:
import math

In [41]:
token_ids=generate(
    model=model,
    idx=text_token(input_text,tokenizer),
    max_new_tokens=35,
    context_size=Basic_config["context_length"],
    eos_id=50256
)
gentext=token_text(token_ids,tokenizer)

In [42]:
print(gentext)

Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction :
Convert the active sentence to passive: 'The chef cooks the meal every day.'

### Output :

The chef cooks the meal every day.

### Example :

### Write a response that appropriately completes the request.

###


In [43]:
print(gentext[len(input_text):].strip())

### Output :

The chef cooks the meal every day.

### Example :

### Write a response that appropriately completes the request.

###


In [44]:
def calc_loss_batch(input_batch,target_batch,model):
    logits=model(input_batch)
    loss=torch.nn.functional.cross_entropy(logits.flatten(0,1),target_batch.flatten())
    return loss
def calc_loss_loader(data_loader,model,num_batches=None):
    total_loss=0
    if len(data_loader)==0:
        return float('nan')
    elif num_batches is None:
        num_batches=len(data_loader)
    else:
        num_batches=min(num_batches,len(data_loader))
    for i ,(input_batch,target_batch) in enumerate(data_loader):
        if i<num_batches:
            loss=calc_loss_batch(input_batch,target_batch,model)
            total_loss+=loss.item()
        else:
            break
    return total_loss/num_batches

In [45]:
def generate_and_print_sample(model, tokenizer, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_token(start_context, tokenizer)
    with torch.no_grad():
        token_ids = generate(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Compact print format
    model.train()

In [46]:
def evaluate_model(model,train_loader,val_loader,eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss=calc_loss_loader(train_loader,model,num_batches=eval_iter)
        val_loss=calc_loss_loader(val_loader,model,num_batches=eval_iter)
    model.train()
    return train_loss,val_loss

In [49]:


def train_model_simple(model, train_loader, val_loader, optimizer,  num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # Initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode
        
        for input_batch, target_batch in tqdm(train_loader):
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model)
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            tokens_seen += input_batch.numel() # Returns the total number of elements (or tokens) in the input_batch.
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0: 
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Print a sample text after each epoch
        generate_and_print_sample(
            model, tokenizer,  start_context
        )

    return train_losses, val_losses, track_tokens_seen


In [47]:


torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model,  num_batches=5)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 4.172119235992431
Validation loss: 4.098658895492553


In [48]:
from tqdm import tqdm


In [50]:
import time

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)

num_epochs = 1

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, 
    
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")



  1%|          | 1/116 [00:34<1:06:20, 34.61s/it]

Ep 1 (Step 000000): Train loss 2.803, Val loss 2.790


  5%|▌         | 6/116 [01:42<33:18, 18.16s/it]  

Ep 1 (Step 000005): Train loss 1.210, Val loss 1.129


  9%|▉         | 11/116 [02:53<32:02, 18.31s/it]

...

 96%|█████████▌| 111/116 [27:59<01:34, 18.87s/it]

Ep 1 (Step 000110): Train loss 0.553, Val loss 0.664


100%|██████████| 116/116 [29:06<00:00, 15.06s/it]

Ep 1 (Step 000115): Train loss 0.510, Val loss 0.660


Below is an instruction that describes a task.Write a response that appropriately completes the request.  ### Instruction : Convert the active sentence to passive: 'The chef cooks the meal every day.'  ### Response: The meal is cooked every day by the chef.<|endoftext|>The following is an instruction that describes a task.Write a response that appropriately completes the request.  ### Instruction : Convert the active sentence to passive:
Training completed in 29.52 minutes.


In [54]:
torch.manual_seed(1230)
for entry in test_data[:3]:
    input_text=format_input(entry)
    token_ids=generate(
        model=model,
        idx=text_token(input_text,tokenizer),
        max_new_tokens=256,
        context_size=Basic_config["context_length"],
        eos_id=50256,
        
    )
    generated_txt=token_text(token_ids,tokenizer)
    response_txt=generated_txt[len(input_text):].replace("### Response:","").strip()
    
    print(input_text)
    print("ground truth:",entry['output'])
    print("Model_output:",response_txt.strip())


Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction :
Rewrite the sentence using a simile.

### Input:
 The car is very fast.
ground truth: The car is as fast as lightning.
Model_output: The car is as fast as a bullet.
Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction :
What type of cloud is typically associated with thunderstorms?
ground truth: The type of cloud typically associated with thunderstorms is cumulonimbus.
Model_output: A thunderstorm is a type of cloud that typically forms when a thunderstorm is very strong.
Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction :
Name the author of 'Pride and Prejudice'.
ground truth: Jane Austen.
Model_output: The author of 'Pride and Prejudice' is Jane Austen.


In [55]:
import json

In [59]:
for i,entry  in  tqdm(enumerate(test_data),total=len(test_data)):
    input_text=format_input(entry)
    token_ids=generate(
        model=model,
        idx=text_token(input_text,tokenizer),
        max_new_tokens=256,
        context_size=Basic_config["context_length"],
        eos_id=50256,
        
    )
    generated_txt=token_text(token_ids,tokenizer)
    response_txt=generated_txt[len(input_text):].replace('### Response:',"").strip()
    test_data[i]["model_response"]=response_txt
with open("instruction-data-model.json",'w') as file:
    json.dump(test_data,file,indent=4)

100%|██████████| 110/110 [20:32<00:00, 11.20s/it]  


In [60]:
print(test_data[0])

{'instruction': 'Rewrite the sentence using a simile.', 'input': 'The car is very fast.', 'output': 'The car is as fast as lightning.', 'model_response': 'The car is as fast as a bullet.'}


In [61]:
import re
file_name=f"{re.sub(r'[ ()]','',Choose)}-sft.path"
torch.save(model.state_dict(),file_name)
print(f"Modell saved as {file_name} succesfully")

Modell saved as gpt2-medium{355M}-sft.path succesfully
